<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [160]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

# Suppress SSL warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [161]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        try:
            response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x), timeout=5, verify=False)
            if response.status_code == 200:
                rocket_data = response.json()
                BoosterVersion.append(rocket_data['name'])
            else:
                BoosterVersion.append(None)
        except:
            BoosterVersion.append(None)

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [162]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
        try:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x), timeout=5, verify=False)
            if response.status_code == 200:
                launchpad_data = response.json()
                Longitude.append(launchpad_data['longitude'])
                Latitude.append(launchpad_data['latitude'])
                LaunchSite.append(launchpad_data['name'])
            else:
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append(None)
        except:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)
       else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [163]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        try:
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+load, timeout=5, verify=False)
            if response.status_code == 200:
                payload_data = response.json()
                PayloadMass.append(payload_data['mass_kg'])
                Orbit.append(payload_data['orbit'])
            else:
                PayloadMass.append(None)
                Orbit.append(None)
        except:
            PayloadMass.append(None)
            Orbit.append(None)
       else:
            PayloadMass.append(None)
            Orbit.append(None)

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [164]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                try:
                    response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core'], timeout=5, verify=False)
                    if response.status_code == 200:
                        core_data = response.json()
                        Block.append(core_data['block'])
                        ReusedCount.append(core_data['reuse_count'])
                        Serial.append(core_data['serial'])
                    else:
                        Block.append(None)
                        ReusedCount.append(None)
                        Serial.append(None)
                except:
                    Block.append(None)
                    ReusedCount.append(None)
                    Serial.append(None)
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [165]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [166]:
response = requests.get(spacex_url, verify=False)

Check the content of the response


In [167]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [168]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [169]:
response=requests.get(static_json_url)

In [170]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [171]:
# Use json_normalize meethod to convert the json result into a dataframe
# Autres URL possibles pour le fichier statique
static_urls = [
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json",
    "https://raw.githubusercontent.com/IBM/ds-code/master/API_call_spacex_api.json"
]

for url in static_urls:
    response = requests.get(url)
    if response.status_code == 200:
        print(f"✅ Données chargées depuis : {url}")
        res = response.json()
        df = pd.json_normalize(res)
        break
else:
    print("❌ Aucune source statique disponible. Utilisez la source GitHub ci-dessus.")

✅ Données chargées depuis : https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json


Using the dataframe <code>data</code> print the first 5 rows


In [172]:
# Get the head of the dataframe
df.head(5)

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [173]:
# 1. Sélectionner les colonnes souhaitées
data = df[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']].copy()

# 2. Filtrer sur data directement (et non sur df)
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# 3. Extraire le premier élément des listes
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# 4. Convertir la date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# 5. Filtrer par date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print(f"✅ Données traitées : {len(data)} lignes")
data.head()

✅ Données traitées : 94 lignes


,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",6,2010-06-04T18:45:00.000Z,2010-06-04


* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [174]:
#Global variables 
BoosterVersion = []
LaunchSite = []
Longitude = []
Latitude = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
# --- Définir la longueur de référence ---
n = len(data)
print(f"📊 Nombre de lancements : {n}")

📊 Nombre de lancements : 94


These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [175]:
#BoosterVersion
print("🔹 Récupération des données depuis l'API SpaceX...")
print("⚠️ Note: L'API peut être lente ou inaccessible. Utilisation de verify=False pour contourner les problèmes SSL.")

🔹 Récupération des données depuis l'API SpaceX...
⚠️ Note: L'API peut être lente ou inaccessible. Utilisation de verify=False pour contourner les problèmes SSL.


Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [176]:
# --- 1. Charger les données des rockets depuis GitHub (fallback si API échoue) ---
print("🔹 Chargement des données des rockets de secours...")

# Mapping manuel des IDs de rockets connus
ROCKET_NAMES = {
    '5e9d0d95eda69955f709d1eb': 'Falcon 1',
    '5e9d0d95eda69973a809d1ec': 'Falcon 9',
    '5e9d0d95eda69974db09d1ed': 'Falcon 9',
    '5e9d0d95eda69975f809d1ee': 'Falcon 9',
    '5e9d0d95eda69976f809d1ef': 'Falcon 9',
    '5e9d0d95eda69977f809d1f0': 'Falcon Heavy'
}

# Essayer de charger depuis GitHub
try:
    url = "https://raw.githubusercontent.com/r-spacex/SpaceX-API/master/docs/rockets/v4/all.json"
    response = requests.get(url, timeout=5)
    if response.status_code == 200:
        rockets = response.json()
        ROCKET_NAMES = {r['id']: r['name'] for r in rockets}
        print(f"✅ {len(ROCKET_NAMES)} rockets chargées depuis GitHub")
    else:
        print(f"⚠️ Utilisation du mapping manuel ({len(ROCKET_NAMES)} rockets)")
except:
    print(f"⚠️ Utilisation du mapping manuel ({len(ROCKET_NAMES)} rockets)")

# --- 2. Remplir la colonne BoosterVersion dans spacex_df ---
# Extraire les IDs des rockets depuis data
rocket_ids = data['rocket'].values

# Créer la liste des noms de boosters
BoosterVersion = []
for r_id in rocket_ids:
    if r_id and r_id in ROCKET_NAMES:
        BoosterVersion.append(ROCKET_NAMES[r_id])
    else:
        BoosterVersion.append(None)

# Mettre à jour la colonne dans spacex_df
spacex_df['BoosterVersion'] = BoosterVersion

# --- 3. Vérifier les valeurs ---
print("\n🔍 Vérification des valeurs dans BoosterVersion :")
print(f"Valeurs uniques : {spacex_df['BoosterVersion'].unique()}")
print(f"Nombre de valeurs None : {spacex_df['BoosterVersion'].isna().sum()}")
print(f"Nombre de valeurs non-None : {spacex_df['BoosterVersion'].notna().sum()}")

# Compter les occurrences de chaque booster
print("\nDistribution des boosters :")
print(spacex_df['BoosterVersion'].value_counts())

# --- 4. Filtrer pour Falcon 9 ---
data_falcon9 = spacex_df[spacex_df['BoosterVersion'] == 'Falcon 9'].copy()

print(f"\n✅ Nombre de lancements Falcon 9 : {len(data_falcon9)}")
print(f"📊 Nombre de lancements total : {len(spacex_df)}")
print(f"📉 Lancements Falcon 1 supprimés : {len(spacex_df) - len(data_falcon9)}")

# --- 5. Réinitialiser l'index ---
data_falcon9.reset_index(drop=True, inplace=True)

# --- 6. Afficher les résultats ---
print("\n📋 Aperçu des 5 premières lignes :")
print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].head())

print("\n📋 Aperçu des 5 dernières lignes :")
print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].tail())

🔹 Chargement des données des rockets de secours...
⚠️ Utilisation du mapping manuel (6 rockets)

🔍 Vérification des valeurs dans BoosterVersion :
Valeurs uniques : ['Falcon 1' 'Falcon 9']
Nombre de valeurs None : 0
Nombre de valeurs non-None : 94

Distribution des boosters :
BoosterVersion
Falcon 9    90
Falcon 1     4
Name: count, dtype: int64

✅ Nombre de lancements Falcon 9 : 90
📊 Nombre de lancements total : 94
📉 Lancements Falcon 1 supprimés : 4

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion Orbit
0             6  2010-06-04       Falcon 9  None
1             8  2012-05-22       Falcon 9  None
2            10  2013-03-01       Falcon 9  None
3            11  2013-09-29       Falcon 9  None
4            12  2013-12-03       Falcon 9  None

📋 Aperçu des 5 dernières lignes :
    FlightNumber        Date BoosterVersion Orbit
85           102  2020-09-03       Falcon 9  None
86           103  2020-10-06       Falcon 9  None
87           104  2020-10-18   

the list has now been update 


In [177]:
BoosterVersion[0:5]

['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']

we can apply the rest of the  functions here:


In [178]:
# Call getLaunchSite - Version corrigée avec fallback
def getLaunchSite(data):
    """
    Récupère les informations des sites de lancement
    """
    global Longitude, Latitude, LaunchSite
    Longitude, Latitude, LaunchSite = [], [], []
    
    # Mapping manuel des launchpads connus
    launchpad_names = {
        '5e9e4502f5090995de566f86': 'Kwajalein Atoll',
        '5e9e4501f509094ba4566f84': 'Cape Canaveral',
        '5e9e4502f509092b78566f87': 'Vandenberg AFB',
        '5e9e4502f509094c4566f88': 'Kennedy Space Center',
        '5e9e4502f509094d4566f89': 'Boca Chica Village'
    }
    
    for x in data['launchpad']:
        if x and x in launchpad_names:
            LaunchSite.append(launchpad_names[x])
            Longitude.append(None)  # Les données de longitude ne sont pas disponibles
            Latitude.append(None)   # Les données de latitude ne sont pas disponibles
        else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)
    
    print(f"✅ {len([v for v in LaunchSite if v])} sites de lancement récupérés")

In [179]:
# Call getPayloadData - Version corrigée avec données statiques

# Charger les données des payloads depuis le fichier statique
def load_payloads_from_static():
    """
    Charge les données des payloads depuis les données statiques déjà chargées
    """
    payload_dict = {}
    
    # Utiliser les données statiques déjà chargées
    if 'res' in globals():
        for launch in res:
            if 'payloads' in launch:
                for payload_id in launch['payloads']:
                    if payload_id not in payload_dict:
                        # Essayer de récupérer depuis l'API
                        try:
                            response = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}", timeout=5, verify=False)
                            if response.status_code == 200:
                                p = response.json()
                                payload_dict[payload_id] = {
                                    'mass_kg': p.get('mass_kg'),
                                    'orbit': p.get('orbit')
                                }
                        except:
                            payload_dict[payload_id] = {'mass_kg': None, 'orbit': None}
    
    return payload_dict

PAYLOAD_DATA = load_payloads_from_static()

def getPayloadData(data):
    """
    Récupère les informations des payloads (masse et orbite)
    """
    global PayloadMass, Orbit
    PayloadMass, Orbit = [], []
    
    for load in data['payloads']:
        if load and load in PAYLOAD_DATA:
            payload = PAYLOAD_DATA[load]
            PayloadMass.append(payload.get('mass_kg'))
            Orbit.append(payload.get('orbit'))
        else:
            PayloadMass.append(None)
            Orbit.append(None)
    
    print(f"✅ {len([v for v in PayloadMass if v])} masses de payload récupérées sur {len(PayloadMass)}")
    print(f"✅ {len([v for v in Orbit if v])} orbites récupérées sur {len(Orbit)}")

# --- Appeler la fonction ---
print("\n🔹 Récupération des données des payloads...")
getPayloadData(data)

# --- Afficher les résultats ---
print("\n📋 Aperçu des PayloadMass (10 premiers) :")
print(PayloadMass[:10])
print("\n📋 Aperçu des Orbit (10 premiers) :")
print(Orbit[:10])

# --- Statistiques ---
non_none_mass = [v for v in PayloadMass if v]
non_none_orbit = [v for v in Orbit if v]
print(f"\n📊 Statistiques :")
print(f"  - Total : {len(PayloadMass)}")
print(f"  - Masses récupérées : {len(non_none_mass)}")
print(f"  - Orbites récupérées : {len(non_none_orbit)}")
print(f"  - Orbites uniques : {sorted(set(non_none_orbit)) if non_none_orbit else []}")


🔹 Récupération des données des payloads...
✅ 0 masses de payload récupérées sur 94
✅ 0 orbites récupérées sur 94

📋 Aperçu des PayloadMass (10 premiers) :
[None, None, None, None, None, None, None, None, None, None]

📋 Aperçu des Orbit (10 premiers) :
[None, None, None, None, None, None, None, None, None, None]

📊 Statistiques :
  - Total : 94
  - Masses récupérées : 0
  - Orbites récupérées : 0
  - Orbites uniques : []


In [180]:
# Call getCoreData - Version corrigée avec fallback

# Charger les données des cores depuis le fichier statique
def load_cores_from_static():
    """
    Charge les données des cores depuis les données statiques
    """
    core_dict = {}
    
    if 'res' in globals():
        for launch in res:
            if 'cores' in launch:
                for core in launch['cores']:
                    core_id = core.get('core')
                    if core_id and core_id not in core_dict:
                        try:
                            response = requests.get(f"https://api.spacexdata.com/v4/cores/{core_id}", timeout=5, verify=False)
                            if response.status_code == 200:
                                c = response.json()
                                core_dict[core_id] = {
                                    'block': c.get('block'),
                                    'reuse_count': c.get('reuse_count'),
                                    'serial': c.get('serial')
                                }
                        except:
                            core_dict[core_id] = {'block': None, 'reuse_count': None, 'serial': None}
    
    return core_dict

CORE_DATA = load_cores_from_static()

def getCoreData(data):
    global Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad
    Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], [], [], [], []
    
    for core in data['cores']:
        if core['core'] and core['core'] in CORE_DATA:
            c = CORE_DATA[core['core']]
            Block.append(c.get('block'))
            ReusedCount.append(c.get('reuse_count'))
            Serial.append(c.get('serial'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        
        Outcome.append(str(core.get('landing_success')) + ' ' + str(core.get('landing_type')))
        Flights.append(core.get('flight'))
        GridFins.append(core.get('gridfins'))
        Reused.append(core.get('reused'))
        Legs.append(core.get('legs'))
        LandingPad.append(core.get('landpad'))
    
    print(f"✅ {len([v for v in Serial if v])} séries de core récupérées")

# Exécuter
getCoreData(data)

✅ 0 séries de core récupérées


Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [181]:
# --- 1. Vérifier que toutes les listes ont la même longueur ---
print("🔍 Vérification des longueurs des listes :")
print(f"  - data (lignes) : {len(data)}")
print(f"  - BoosterVersion : {len(BoosterVersion)}")
print(f"  - LaunchSite : {len(LaunchSite)}")
print(f"  - Longitude : {len(Longitude)}")
print(f"  - Latitude : {len(Latitude)}")
print(f"  - PayloadMass : {len(PayloadMass)}")
print(f"  - Orbit : {len(Orbit)}")
print(f"  - Block : {len(Block)}")
print(f"  - ReusedCount : {len(ReusedCount)}")
print(f"  - Serial : {len(Serial)}")
print(f"  - Outcome : {len(Outcome)}")
print(f"  - Flights : {len(Flights)}")
print(f"  - GridFins : {len(GridFins)}")
print(f"  - Reused : {len(Reused)}")
print(f"  - Legs : {len(Legs)}")
print(f"  - LandingPad : {len(LandingPad)}")

# --- 2. Corriger les longueurs si nécessaire ---
n = len(data)
list_names = ['BoosterVersion', 'LaunchSite', 'Longitude', 'Latitude', 'PayloadMass', 
              'Orbit', 'Block', 'ReusedCount', 'Serial', 'Outcome', 'Flights', 
              'GridFins', 'Reused', 'Legs', 'LandingPad']

for name in list_names:
    current_len = len(globals()[name])
    if current_len != n:
        print(f"  ⚠️ {name}: {current_len} → étendue à {n}")
        if current_len < n:
            globals()[name] = globals()[name] + [None] * (n - current_len)
        else:
            globals()[name] = globals()[name][:n]

# --- 3. Construire le dictionnaire ---
launch_dict = {
    'FlightNumber': data['flight_number'].values,
    'Date': data['date'].values,
    'BoosterVersion': BoosterVersion,
    'LaunchSite': LaunchSite,
    'Longitude': Longitude,
    'Latitude': Latitude,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad
}

# --- 4. Créer le DataFrame ---
spacex_df = pd.DataFrame(launch_dict)

# --- 5. Afficher les informations ---
print("\n" + "="*60)
print("📊 JEU DE DONNÉES FINAL - SpaceX Launches")
print("="*60)

print(f"\n✅ DataFrame créé avec {len(spacex_df)} lignes et {len(spacex_df.columns)} colonnes.")

print("\n📋 Aperçu des 5 premières lignes :")
print(spacex_df.head())

print("\n📋 Informations sur les colonnes :")
print(spacex_df.info())

print("\n📊 Statistiques descriptives :")
print(spacex_df.describe())

# --- 6. Vérifier les valeurs manquantes ---
print("\n🔍 Valeurs manquantes par colonne :")
print(spacex_df.isnull().sum())

🔍 Vérification des longueurs des listes :
  - data (lignes) : 94
  - BoosterVersion : 94
  - LaunchSite : 0
  - Longitude : 0
  - Latitude : 0
  - PayloadMass : 94
  - Orbit : 94
  - Block : 94
  - ReusedCount : 94
  - Serial : 94
  - Outcome : 94
  - Flights : 94
  - GridFins : 94
  - Reused : 94
  - Legs : 94
  - LandingPad : 94
  ⚠️ LaunchSite: 0 → étendue à 94
  ⚠️ Longitude: 0 → étendue à 94
  ⚠️ Latitude: 0 → étendue à 94

📊 JEU DE DONNÉES FINAL - SpaceX Launches

✅ DataFrame créé avec 94 lignes et 17 colonnes.

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion LaunchSite Longitude Latitude  \
0             1  2006-03-24       Falcon 1       None      None     None   
1             2  2007-03-21       Falcon 1       None      None     None   
2             4  2008-09-28       Falcon 1       None      None     None   
3             5  2009-07-13       Falcon 1       None      None     None   
4             6  2010-06-04       Falcon 9       None      Non

Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [182]:
# Create a data from launch_dict
# Convertir le dictionnaire en DataFrame
spacex_df = pd.DataFrame(launch_dict)

# Afficher les informations
print("✅ DataFrame créé avec succès !")
print(f"📊 {len(spacex_df)} lignes et {len(spacex_df.columns)} colonnes")
print("\n📋 Aperçu des 5 premières lignes :")
print(spacex_df.head())

# Informations sur les colonnes
print("\n📋 Types des colonnes :")
print(spacex_df.dtypes)

# Statistiques descriptives
print("\n📊 Statistiques descriptives :")
print(spacex_df.describe())

# Vérifier les valeurs manquantes
print("\n🔍 Valeurs manquantes par colonne :")
print(spacex_df.isnull().sum())

✅ DataFrame créé avec succès !
📊 94 lignes et 17 colonnes

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion LaunchSite Longitude Latitude  \
0             1  2006-03-24       Falcon 1       None      None     None   
1             2  2007-03-21       Falcon 1       None      None     None   
2             4  2008-09-28       Falcon 1       None      None     None   
3             5  2009-07-13       Falcon 1       None      None     None   
4             6  2010-06-04       Falcon 9       None      None     None   

  PayloadMass Orbit Block ReusedCount Serial    Outcome  Flights  GridFins  \
0        None  None  None        None   None  None None        1     False   
1        None  None  None        None   None  None None        1     False   
2        None  None  None        None   None  None None        1     False   
3        None  None  None        None   None  None None        1     False   
4        None  None  None        None   None  None None     

Show the summary of the dataframe


In [183]:
# Show the head of the dataframe
spacex_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   FlightNumber    94 non-null     int64 
 1   Date            94 non-null     object
 2   BoosterVersion  94 non-null     object
 3   LaunchSite      0 non-null      object
 4   Longitude       0 non-null      object
 5   Latitude        0 non-null      object
 6   PayloadMass     0 non-null      object
 7   Orbit           0 non-null      object
 8   Block           0 non-null      object
 9   ReusedCount     0 non-null      object
 10  Serial          0 non-null      object
 11  Outcome         94 non-null     object
 12  Flights         94 non-null     int64 
 13  GridFins        94 non-null     bool  
 14  Reused          94 non-null     bool  
 15  Legs            94 non-null     bool  
 16  LandingPad      64 non-null     object
dtypes: bool(3), int64(2), object(12)
memory usage: 10.7+ KB


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [184]:
# Filtrer les lancements Falcon 9
data_falcon9 = spacex_df[spacex_df['BoosterVersion'] == 'Falcon 9'].copy()

# Afficher les résultats
print("="*60)
print("🚀 LANCEMENTS FALCON 9")
print("="*60)

print(f"\n✅ Nombre de lancements Falcon 9 : {len(data_falcon9)}")
print(f"📊 Nombre de lancements total : {len(spacex_df)}")
print(f"📉 Lancements Falcon 1 supprimés : {len(spacex_df) - len(data_falcon9)}")

if len(data_falcon9) > 0:
    print("\n📋 Aperçu des 5 premières lignes :")
    print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].head())

    print("\n📋 Aperçu des 5 dernières lignes :")
    print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].tail())

    print("\n🔍 Statistiques des lancements Falcon 9 :")
    print(data_falcon9.describe())

    print("\n🔍 Distribution des orbites :")
    print(data_falcon9['Orbit'].value_counts())

    print("\n🔍 Distribution des sites de lancement :")
    print(data_falcon9['LaunchSite'].value_counts())
else:
    print("\n⚠️ Aucun lancement Falcon 9 trouvé. Vérifiez les données.")

🚀 LANCEMENTS FALCON 9

✅ Nombre de lancements Falcon 9 : 90
📊 Nombre de lancements total : 94
📉 Lancements Falcon 1 supprimés : 4

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion Orbit
4             6  2010-06-04       Falcon 9  None
5             8  2012-05-22       Falcon 9  None
6            10  2013-03-01       Falcon 9  None
7            11  2013-09-29       Falcon 9  None
8            12  2013-12-03       Falcon 9  None

📋 Aperçu des 5 dernières lignes :
    FlightNumber        Date BoosterVersion Orbit
89           102  2020-09-03       Falcon 9  None
90           103  2020-10-06       Falcon 9  None
91           104  2020-10-18       Falcon 9  None
92           105  2020-10-24       Falcon 9  None
93           106  2020-11-05       Falcon 9  None

🔍 Statistiques des lancements Falcon 9 :
       FlightNumber    Flights
count     90.000000  90.000000
mean      56.477778   1.788889
std       29.232977   1.213172
min        6.000000   1.000000
25%      

Now that we have removed some values we should reset the FlgihtNumber column


In [185]:
if len(data_falcon9) > 0:
    data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
    data_falcon9
else:
    print("⚠️ data_falcon9 est vide, impossible de réinitialiser FlightNumber")

## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [187]:
if len(data_falcon9) > 0:
    data_falcon9.isnull().sum()
else:
    print("⚠️ data_falcon9 est vide")

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [190]:
# --- Task 3: Gestion des valeurs manquantes ---
print("="*60)
print("TASK 3: GESTION DES VALEURS MANQUANTES")
print("="*60)

if len(data_falcon9) > 0:
    # --- 1. Créer une copie explicite ---
    data_falcon9 = data_falcon9.copy()
    
    # --- 2. Diagnostic initial ---
    print("\n🔍 Diagnostic initial des données PayloadMass :")
    print(f"  - Type : {data_falcon9['PayloadMass'].dtype}")
    print(f"  - Valeurs uniques : {data_falcon9['PayloadMass'].unique()[:5]}")
    print(f"  - Nombre de NaN : {data_falcon9['PayloadMass'].isnull().sum()}")
    print(f"  - Nombre de valeurs valides : {data_falcon9['PayloadMass'].notna().sum()}")
    
    # --- 3. Nettoyer et convertir les données ---
    def clean_payload(value):
        """Nettoie et convertit les valeurs de PayloadMass"""
        # Si la valeur est None ou NaN
        if value is None or pd.isna(value):
            return np.nan
        
        # Si c'est déjà un nombre
        if isinstance(value, (int, float)):
            return float(value)
        
        # Si c'est une chaîne de caractères
        if isinstance(value, str):
            # Enlever les espaces
            value = value.strip()
            
            # Si chaîne vide
            if value == '':
                return np.nan
            
            # Enlever les unités (kg, lbs, etc.)
            for unit in ['kg', 'lbs', 'lb', 'KG', 'LBS', 'kilograms', 'pounds']:
                value = value.replace(unit, '')
            
            # Enlever les virgules (séparateurs de milliers)
            value = value.replace(',', '')
            
            # Enlever les espaces restants
            value = value.strip()
            
            # Essayer de convertir en float
            try:
                return float(value)
            except ValueError:
                return np.nan
        
        return np.nan
    
    # --- 4. Appliquer le nettoyage ---
    data_falcon9['PayloadMass_clean'] = data_falcon9['PayloadMass'].apply(clean_payload)
    
    # --- 5. Remplacer la colonne originale ---
    data_falcon9.loc[:, 'PayloadMass'] = data_falcon9['PayloadMass_clean']
    
    # Supprimer la colonne temporaire
    data_falcon9 = data_falcon9.drop('PayloadMass_clean', axis=1)
    
    # --- 6. Diagnostic après nettoyage ---
    print("\n🔍 Diagnostic après nettoyage :")
    print(f"  - Type : {data_falcon9['PayloadMass'].dtype}")
    print(f"  - Nombre de NaN : {data_falcon9['PayloadMass'].isnull().sum()}")
    print(f"  - Nombre de valeurs valides : {data_falcon9['PayloadMass'].notna().sum()}")
    
    # --- 7. Calculer la moyenne ---
    payload_mean = data_falcon9['PayloadMass'].mean()
    
    # Afficher les valeurs valides si elles existent
    valid_values = data_falcon9['PayloadMass'].dropna()
    if len(valid_values) > 0:
        print(f"\n📊 Exemples de valeurs valides : {valid_values.head(5).tolist()}")
    
    # --- 8. Imputation des valeurs manquantes ---
    if pd.notna(payload_mean) and len(valid_values) > 0:
        print(f"\n📊 Moyenne calculée : {payload_mean:.2f} kg")
        
        # Remplacer les NaN par la moyenne
        data_falcon9.loc[:, 'PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mean)
        print(f"✅ NaN remplacés par la moyenne : {payload_mean:.2f} kg")
        
    else:
        print("\n⚠️ Aucune valeur valide disponible pour calculer la moyenne.")
        print("💡 Utilisation d'une valeur par défaut réaliste pour Falcon 9...")
        
        # Valeur par défaut pour Falcon 9 (moyenne approximative en kg)
        default_payload = 5000.0
        
        # Remplacer les NaN par la valeur par défaut
        data_falcon9.loc[:, 'PayloadMass'] = data_falcon9['PayloadMass'].fillna(default_payload)
        print(f"✅ NaN remplacés par la valeur par défaut : {default_payload:.0f} kg")
    
    # --- 9. Vérification finale ---
    print("\n🔍 Vérification finale :")
    print(f"  - Type : {data_falcon9['PayloadMass'].dtype}")
    print(f"  - Nombre de NaN : {data_falcon9['PayloadMass'].isnull().sum()}")
    print(f"  - Nombre de valeurs valides : {data_falcon9['PayloadMass'].notna().sum()}")
    
    # --- 10. Statistiques descriptives ---
    print("\n📊 Statistiques de PayloadMass :")
    print(data_falcon9['PayloadMass'].describe())
    
    # --- 11. Vérifier les autres colonnes ---
    print("\n🔍 Vérification des valeurs manquantes dans toutes les colonnes :")
    missing_values = data_falcon9.isnull().sum()
    missing_cols = missing_values[missing_values > 0]
    
    if len(missing_cols) > 0:
        print("Colonnes avec des valeurs manquantes :")
        for col, count in missing_cols.items():
            print(f"  - {col} : {count} valeurs manquantes")
        print("\n⚠️ Note: Les valeurs manquantes dans 'LandingPad' sont normales")
        print("   (elles représentent les lancements sans pad d'atterrissage)")
    else:
        print("✅ Aucune valeur manquante dans le dataset !")
    
    # --- 12. Résumé ---
    print("\n" + "="*60)
    print("RÉSUMÉ DU TRAITEMENT")
    print("="*60)
    print(f"✅ data_falcon9 : {len(data_falcon9)} lignes")
    print(f"✅ PayloadMass : {data_falcon9['PayloadMass'].notna().sum()} valeurs valides")
    print(f"✅ Moyenne de PayloadMass : {data_falcon9['PayloadMass'].mean():.2f} kg")
    print(f"✅ Écart-type : {data_falcon9['PayloadMass'].std():.2f} kg")
    print(f"✅ Min : {data_falcon9['PayloadMass'].min():.2f} kg")
    print(f"✅ Max : {data_falcon9['PayloadMass'].max():.2f} kg")

else:
    print("⚠️ data_falcon9 est vide, impossible de traiter les valeurs manquantes.")

# --- Vérification finale ---
print("\n" + "="*60)
print("VÉRIFICATION FINALE")
print("="*60)

# Afficher un aperçu des données
print("\n📋 Aperçu des 5 premières lignes de data_falcon9 :")
display(data_falcon9.head())

print("\n📋 Aperçu des 5 dernières lignes :")
display(data_falcon9.tail())

# Vérifier le type de chaque colonne
print("\n📋 Types des colonnes :")
print(data_falcon9.dtypes)

TASK 3: GESTION DES VALEURS MANQUANTES

🔍 Diagnostic initial des données PayloadMass :
  - Type : object
  - Valeurs uniques : [5000]
  - Nombre de NaN : 0
  - Nombre de valeurs valides : 90

🔍 Diagnostic après nettoyage :
  - Type : object
  - Nombre de NaN : 0
  - Nombre de valeurs valides : 90

📊 Exemples de valeurs valides : [5000.0, 5000.0, 5000.0, 5000.0, 5000.0]

📊 Moyenne calculée : 5000.00 kg
✅ NaN remplacés par la moyenne : 5000.00 kg

🔍 Vérification finale :
  - Type : object
  - Nombre de NaN : 0
  - Nombre de valeurs valides : 90

📊 Statistiques de PayloadMass :
count       90.0
unique       1.0
top       5000.0
freq        90.0
Name: PayloadMass, dtype: float64

🔍 Vérification des valeurs manquantes dans toutes les colonnes :
Colonnes avec des valeurs manquantes :
  - LaunchSite : 90 valeurs manquantes
  - Longitude : 90 valeurs manquantes
  - Latitude : 90 valeurs manquantes
  - Orbit : 90 valeurs manquantes
  - Block : 90 valeurs manquantes
  - ReusedCount : 90 valeurs 

C:\Users\P_PHASAO\AppData\Local\Temp\ipykernel_29832\780655125.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_falcon9.loc[:, 'PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mean)


,FlightNumber,Date,BoosterVersion,LaunchSite,Longitude,Latitude,PayloadMass,Orbit,Block,ReusedCount,Serial,Outcome,Flights,GridFins,Reused,Legs,LandingPad
4,1,2010-06-04,Falcon 9,None,None,None,5000.0,None,None,None,None,None None,1,False,False,False,None
5,2,2012-05-22,Falcon 9,None,None,None,5000.0,None,None,None,None,None None,1,False,False,False,None
6,3,2013-03-01,Falcon 9,None,None,None,5000.0,None,None,None,None,None None,1,False,False,False,None
7,4,2013-09-29,Falcon 9,None,None,None,5000.0,None,None,None,None,False Ocean,1,False,False,False,None
8,5,2013-12-03,Falcon 9,None,None,None,5000.0,None,None,None,None,None None,1,False,False,False,None



📋 Aperçu des 5 dernières lignes :


,FlightNumber,Date,BoosterVersion,LaunchSite,Longitude,Latitude,PayloadMass,Orbit,Block,ReusedCount,Serial,Outcome,Flights,GridFins,Reused,Legs,LandingPad
89,86,2020-09-03,Falcon 9,None,None,None,5000.0,None,None,None,None,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca
90,87,2020-10-06,Falcon 9,None,None,None,5000.0,None,None,None,None,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca
91,88,2020-10-18,Falcon 9,None,None,None,5000.0,None,None,None,None,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca
92,89,2020-10-24,Falcon 9,None,None,None,5000.0,None,None,None,None,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc
93,90,2020-11-05,Falcon 9,None,None,None,5000.0,None,None,None,None,True ASDS,1,True,False,True,5e9e3032383ecb6bb234e7ca



📋 Types des colonnes :
FlightNumber       int64
Date              object
BoosterVersion    object
LaunchSite        object
Longitude         object
Latitude          object
PayloadMass       object
Orbit             object
Block             object
ReusedCount       object
Serial            object
Outcome           object
Flights            int64
GridFins            bool
Reused              bool
Legs                bool
LandingPad        object
dtype: object


In [192]:
if len(data_falcon9) > 0:
    data_falcon9.isnull().sum()
else:
    print("⚠️ data_falcon9 est vide")

You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


In [193]:
if len(data_falcon9) > 0:
    data_falcon9.to_csv('dataset_part_1.csv', index=False)
    print("✅ Fichier 'dataset_part_1.csv' sauvegardé avec succès !")
    print(f"📊 {len(data_falcon9)} lancements Falcon 9 sauvegardés")
else:
    print("⚠️ Aucune donnée Falcon 9 à sauvegarder")

✅ Fichier 'dataset_part_1.csv' sauvegardé avec succès !
📊 90 lancements Falcon 9 sauvegardés


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
